# PM Assistant v3 — Multi-Project Edition
**What this notebook does:**
1. **Scoper** → Breaks project into tasks
2. **Mapper** → Maps task dependencies
3. **Scheduler** → Assigns start/end days
4. **Allocator** → Assigns tasks to team members
5. **Auditor** → Scores risk (0-10 per task)
6. **Optimizer** → Suggests fixes if risk is too high *(loops back)*
7. **Summarizer** → Generates a plain-English summary + improvement suggestions

## Imports & Setup

In [1]:
import os
import uuid
import pandas as pd
import plotly.express as px
from datetime import datetime, timedelta
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from IPython.display import display, Markdown

# Load API key from .env file (recommended) or set directly
load_dotenv()
# If you don't have a .env file, uncomment the line below and paste your key:
os.environ['GROQ_API_KEY'] = "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"

llm = ChatOllama(model="llama3.2", temperature=0.1)
test = llm.invoke("Say hello in one word")
print(f"Ollama connected! Response: {test.content}")

Ollama connected! Response: Hello.


## Data Models

In [2]:
from typing import List, Optional, Any, Annotated
from pydantic import BaseModel, Field, AliasChoices, BeforeValidator, model_validator

# --- Helpers ---
def force_string(v):
    return str(v) if v is not None else None
StringId = Annotated[Optional[str], BeforeValidator(force_string)]

# --- Core Entities ---
class Task(BaseModel):
    id: StringId = None
    task_name: str = Field(..., validation_alias=AliasChoices("title", "name", "task_name", "task"))
    task_description: str = Field(default="", validation_alias=AliasChoices("description", "task_description", "desc"))
    estimated_day: int = Field(..., validation_alias=AliasChoices("duration_days", "estimated_days", "estimated_day", "duration", "estimated_time", "time"))
    required_skill: str = Field(default="General", validation_alias=AliasChoices("required_skill", "skill", "skills"))

class TeamMember(BaseModel):
    name: str
    role: str
    skills: List[str]
    seniority: str 

class Team(BaseModel):
    team_members: List[TeamMember]

class TaskList(BaseModel):
    task: List[Task] = Field(..., validation_alias=AliasChoices("tasks", "task"))
    
    @model_validator(mode='before')
    @classmethod
    def wrap_list(cls, data: Any) -> Any:
        if isinstance(data, list): return {"task": data}
        if isinstance(data, dict) and "tasks" in data: return {"task": data["tasks"]}
        return data

class Dependency(BaseModel):
    task_id: str = Field(..., validation_alias=AliasChoices("task_id", "id", "task"))
    dependent_on: List[str] = Field(..., validation_alias=AliasChoices("dependent_on", "dependencies", "deps", "depends_on"))

class DependencyList(BaseModel):
    dependencies: List[Dependency] = Field(..., validation_alias=AliasChoices("dependencies", "deps"))
    
    @model_validator(mode='before')
    @classmethod
    def wrap_list(cls, data: Any) -> Any:
        if isinstance(data, list): return {"dependencies": data}
        if isinstance(data, dict):
            deps = data.get("dependencies", data.get("deps"))
            if isinstance(deps, dict):
                converted = [{"task_id": k, "dependent_on": v} for k, v in deps.items()]
                return {"dependencies": converted}
            if isinstance(deps, list):
                return {"dependencies": deps}
        return data

class TaskSchedule(BaseModel):
    task: Task
    start_day: int
    end_day: int

class Schedule(BaseModel):
    schedule: List[TaskSchedule]

class TaskAllocation(BaseModel):
    task: Task
    team_member: TeamMember

class TaskAllocationList(BaseModel):
    task_allocations: List[TaskAllocation]
    
    @model_validator(mode='before')
    @classmethod
    def wrap_list(cls, data: Any) -> Any:
        if isinstance(data, list): return {"task_allocations": data}
        if isinstance(data, dict):
            payload = data.get("task_allocations") or data.get("allocs") or data.get("allocations")
            if isinstance(payload, dict):
                 converted = [{"task_id": k, "member_name": v} for k, v in payload.items()]
                 return {"task_allocations": converted}
            if isinstance(payload, list):
                return {"task_allocations": payload}
        return data

class Risk(BaseModel):
    task_name: str
    score: int
    reason: str

class RiskList(BaseModel):
    risks: List[Risk]

print("Data models loaded!")

Data models loaded!


## Agent State

In [3]:
from typing import TypedDict

class AgentState(TypedDict):
    project_description: str
    team: Team
    tasks: TaskList
    dependencies: List[dict]
    schedule: Schedule
    task_allocations: TaskAllocationList
    risks: RiskList
    iteration_number: int
    max_iteration: int
    insights: List[str]
    project_risk_score_iterations: List[int]
    # New field for the final summary
    final_summary: str

print("AgentState defined!")

AgentState defined!


## Node Logic (Agents)

In [4]:
# ─────────────────────────────────────────────
# NODE 1: Scoper — breaks project into tasks
# ─────────────────────────────────────────────
def scope_decomposition_node(state: AgentState):
    print("Node: Scoper — decomposing project into tasks...")
    prompt = f"""
    Project: {state['project_description']}
    Team Skills Available: {[m.skills for m in state['team'].team_members]}
    
    Break this project into granular tasks (max 4 days per task).
    
    CRITICAL INSTRUCTIONS:
    1. Return a JSON object matching the TaskList schema.
    2. REQUIRED FIELDS: 'task_name', 'task_description', 'estimated_day', 'required_skill'.
    3. Do NOT assign names (like Alice or Bob). Only identify the 'required_skill'.
    
    Example Output:
    {{
      "tasks": [
        {{
            "task_name": "Setup Repo", 
            "task_description": "Initialize Git repo and CI/CD pipelines", 
            "estimated_day": 1, 
            "required_skill": "DevOps"
        }}
      ]
    }}
    """
    struct_llm = llm.with_structured_output(TaskList, method="json_mode")
    response = struct_llm.invoke(prompt)
    
    for t in response.task:
        if not t.id: t.id = str(uuid.uuid4())[:4]
        
    print(f" {len(response.task)} tasks created")
    return {"tasks": response}


# ─────────────────────────────────────────────
# NODE 2: Mapper — maps dependencies between tasks
# ─────────────────────────────────────────────
def dependency_mapping_node(state: AgentState):
    print("🔗 Node: Mapper — identifying task dependencies...")
    tasks_fmt = "\n".join([f"ID: {t.id} | Name: {t.task_name}" for t in state['tasks'].task])
    
    prompt = f"""
    Map dependencies for these tasks:
    {tasks_fmt}
    
    Return a JSON object matching DependencyList. 
    Use IDs to reference tasks. 
    
    Example Output:
    {{
      "dependencies": [
         {{"task_id": "1a2b", "dependent_on": []}},
         {{"task_id": "3c4d", "dependent_on": ["1a2b"]}}
      ]
    }}
    """
    struct_llm = llm.with_structured_output(DependencyList, method="json_mode")
    response = struct_llm.invoke(prompt)
    print(f"   {len(response.dependencies)} dependencies mapped")
    return {"dependencies": response.dependencies}


# ─────────────────────────────────────────────
# NODE 3: Scheduler — assigns start/end days
# ─────────────────────────────────────────────
def smart_scheduler_node(state: AgentState):
    print("Node: Scheduler — building timeline...")
    
    class SimpleSchedItem(BaseModel):
        task_id: str = Field(..., validation_alias=AliasChoices("task_id", "id", "task_name", "task"))
        start: int = Field(..., validation_alias=AliasChoices("start", "start_day"))
        end: int = Field(..., validation_alias=AliasChoices("end", "end_day"))
    
    class SimpleSched(BaseModel):
        items: List[SimpleSchedItem]
        
        @model_validator(mode='before')
        @classmethod
        def wrap(cls, data):
            if isinstance(data, list): return {"items": data}
            if isinstance(data, dict):
                if "tasks" in data: return {"items": data["tasks"]}
                if "schedule" in data: return {"items": data["schedule"]}
                if "timeline" in data: return {"items": data["timeline"]}
            return data

    prompt = f"""
    Schedule these tasks (Tasks: {state['tasks']}) 
    considering Dependencies: {state.get('dependencies')}
    Previous Insights: {state.get('insights', [])}
    
    Return JSON with start/end days (integers).
    """
    struct_llm = llm.with_structured_output(SimpleSched, method="json_mode")
    resp = struct_llm.invoke(prompt)
    
    task_map = {t.id: t for t in state['tasks'].task}
    name_map = {t.task_name: t for t in state['tasks'].task}
    
    final_sched = []
    for item in resp.items:
        task = task_map.get(str(item.task_id)) or name_map.get(str(item.task_id))
        if task:
            final_sched.append(TaskSchedule(task=task, start_day=item.start, end_day=item.end))
            
    print(f"   {len(final_sched)} tasks scheduled")
    return {"schedule": Schedule(schedule=final_sched)}


# ─────────────────────────────────────────────
# NODE 4: Allocator — assigns tasks to people
# ─────────────────────────────────────────────
def resource_allocation_node(state: AgentState):
    print("👥 Node: Allocator — assigning tasks to team members...")
    
    class SimpleAllocItem(BaseModel):
        task_id: str = Field(..., validation_alias=AliasChoices("task_id", "id", "task_name", "task"))
        member_name: str = Field(..., validation_alias=AliasChoices("member_name", "member", "assignee", "person", "name"))
        
    class SimpleAlloc(BaseModel):
        allocs: List[SimpleAllocItem]
        
        @model_validator(mode='before')
        @classmethod
        def wrap(cls, data):
            if isinstance(data, list): return {"allocs": data}
            if isinstance(data, dict):
                if "allocations" in data: return {"allocs": data["allocations"]}
                if "task_allocations" in data: return {"allocs": data["task_allocations"]}
                if "assignments" in data: return {"allocs": data["assignments"]}
                if "team" in data: return {"allocs": data["team"]}
            return data

    prompt = f"""
    Allocate tasks: {state['tasks']}
    To Team: {state['team']}
    
    Match tasks to team members based on required skills and seniority.
    IMPORTANT: Return JSON.
    
    Output Format Example:
    {{
      "allocs": [
        {{"task_id": "1a2b", "member_name": "Alice"}},
        {{"task_id": "3c4d", "member_name": "Bob"}}
      ]
    }}
    """
    struct_llm = llm.with_structured_output(SimpleAlloc, method="json_mode")
    resp = struct_llm.invoke(prompt)
    
    task_map = {t.id: t for t in state['tasks'].task}
    name_map = {t.task_name: t for t in state['tasks'].task}
    member_map = {m.name: m for m in state['team'].team_members}
    
    final_allocs = []
    for a in resp.allocs:
        task = task_map.get(str(a.task_id)) or name_map.get(str(a.task_id))
        member = member_map.get(a.member_name)
        if task and member:
            final_allocs.append(TaskAllocation(task=task, team_member=member))
            
    print(f"   {len(final_allocs)} tasks allocated")
    return {"task_allocations": TaskAllocationList(task_allocations=final_allocs)}


# ─────────────────────────────────────────────
# NODE 5: Auditor — scores risk per task
# ─────────────────────────────────────────────
def risk_audit_node(state: AgentState):
    print("Node: Auditor — calculating risk scores...")
    
    def repair_risk_item(item: Any) -> Any:
        if isinstance(item, dict):
            if "reason" not in item:
                item["reason"] = item.get("description", item.get("justification", "No reason provided"))
            if "task_name" not in item and "task_id" in item:
                item["task_name"] = str(item["task_id"])
            if "score" not in item:
                item["score"] = 5
        return item

    class SimpleRisk(BaseModel):
        task_name: str = Field(..., validation_alias=AliasChoices("task_name", "task", "name", "Task Name", "Task"))
        score: int = Field(..., validation_alias=AliasChoices("score", "risk_score", "Risk Score"))
        reason: str = Field(..., validation_alias=AliasChoices("reason", "description", "justification", "Reason", "explanation"))
        
        @model_validator(mode='before')
        @classmethod
        def fix_data(cls, data):
            return repair_risk_item(data)

    class SimpleRiskList(BaseModel):
        risks: List[SimpleRisk]
        
        @model_validator(mode='before')
        @classmethod
        def wrap(cls, data):
            if isinstance(data, dict):
                for key in ["risks", "issues", "threats", "audit", "High-Risk Tasks", "tasks", "Risk Breakdown", "recommendations"]:
                    if key in data and isinstance(data[key], list):
                        return {"risks": data[key]}
                for val in data.values():
                    if isinstance(val, list) and len(val) > 0 and isinstance(val[0], dict):
                        return {"risks": val}
            if isinstance(data, list): return {"risks": data}
            return data

    prompt = f"""
    Audit this Plan for Risks (0-10 score):
    Schedule: {state.get('schedule')}
    Allocations: {state.get('task_allocations')}
    
    IMPORTANT: Return a JSON object with a single key "risks" containing a list of objects.
    Each object MUST have: "task_name", "score", and "reason".
    """
    struct_llm = llm.with_structured_output(SimpleRiskList, method="json_mode")
    resp = struct_llm.invoke(prompt)
    
    final_risks = [Risk(task_name=r.task_name, score=r.score, reason=r.reason) for r in resp.risks]
    risks_obj = RiskList(risks=final_risks)
    
    score = sum(r.score for r in final_risks)
    print(f"   Total risk score: {score}")
    return {
        "risks": risks_obj,
        "project_risk_score_iterations": state.get('project_risk_score_iterations', []) + [score],
        "iteration_number": state.get('iteration_number', 0) + 1
    }


# ─────────────────────────────────────────────
# NODE 6: Optimizer — suggests risk reductions
# ─────────────────────────────────────────────
def optimization_insight_node(state: AgentState):
    print("💡 Node: Optimizer — generating improvement suggestions...")
    prompt = f"Risks: {state['risks']}. Suggest 1 concrete schedule/allocation change to lower risk."
    insight = llm.invoke(prompt).content
    print(f"   Insight generated")
    return {"insights": state.get('insights', []) + [insight]}


# ─────────────────────────────────────────────────────────────────
# NODE 7: Summarizer — plain-English summary + code improvements
# ─────────────────────────────────────────────────────────────────
def project_summary_node(state: AgentState):
    print("Node: Summarizer — writing final project summary...")
    
    # Build a compact representation of the final plan
    task_list = "\n".join(
        [f"  - {t.task_name} ({t.estimated_day}d, skill: {t.required_skill})" 
         for t in state['tasks'].task]
    ) if state.get('tasks') else "No tasks"
    
    alloc_list = "\n".join(
        [f"  - {a.task.task_name} → {a.team_member.name} ({a.team_member.role})"
         for a in state['task_allocations'].task_allocations]
    ) if state.get('task_allocations') else "No allocations"
    
    risk_list = "\n".join(
        [f"  - [{r.score}/10] {r.task_name}: {r.reason}"
         for r in sorted(state['risks'].risks, key=lambda x: x.score, reverse=True)]
    ) if state.get('risks') else "No risks"
    
    risk_history = state.get('project_risk_score_iterations', [])
    risk_trend = "improved" if len(risk_history) > 1 and risk_history[-1] < risk_history[0] else "stable"
    
    prompt = f"""
    You are a senior PM. Write a clear, concise project summary report based on this data:

    PROJECT: {state['project_description']}

    TEAM:
    {chr(10).join([f'  - {m.name} ({m.role}, {m.seniority})' for m in state['team'].team_members])}

    TASKS BREAKDOWN:
    {task_list}

    TASK ASSIGNMENTS:
    {alloc_list}

    RISK ASSESSMENT (sorted by severity):
    {risk_list}

    RISK SCORE HISTORY: {risk_history} ({risk_trend})

    OPTIMIZER INSIGHTS:
    {chr(10).join(state.get('insights', ['None']))}

    Please write a Markdown report with these sections:
    1. **Executive Summary** (2-3 sentences, plain English)
    2. **Key Risks & Mitigations** (top 3 risks with action items)
    3. **Team Workload Balance** (who has too much / too little)
    4. **Recommended Next Steps** (3 concrete actions the PM should take)
    5. **Code & Architecture Improvements** (suggest 3 improvements to the LangGraph agent code itself)
    """
    
    summary = llm.invoke(prompt).content
    print("   Summary complete!")
    return {"final_summary": summary}


print("All 7 nodes defined!")

All 7 nodes defined!


## Graph Structure

```
scoper → mapper → scheduler → allocator → auditor
                     ↑                       ↓
                  optimizer ←── (if risk high)
                                        ↓ (if risk low or max iterations)
                                   summarizer → END
```

In [5]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def routing_logic(state: AgentState):
    """After auditing: loop to optimizer if risk is high, else go to summarizer."""
    last_score = state['project_risk_score_iterations'][-1]
    if state["iteration_number"] >= state["max_iteration"] or last_score < 15:
        return "summarizer"  # Go to summarizer instead of END
    return "optimizer"

workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("scoper", scope_decomposition_node)
workflow.add_node("mapper", dependency_mapping_node)
workflow.add_node("scheduler", smart_scheduler_node)
workflow.add_node("allocator", resource_allocation_node)
workflow.add_node("auditor", risk_audit_node)
workflow.add_node("optimizer", optimization_insight_node)
workflow.add_node("summarizer", project_summary_node)  # 🆕

# Add Edges
workflow.set_entry_point("scoper")
workflow.add_edge("scoper", "mapper")
workflow.add_edge("mapper", "scheduler")
workflow.add_edge("scheduler", "allocator")
workflow.add_edge("allocator", "auditor")
workflow.add_conditional_edges("auditor", routing_logic)
workflow.add_edge("optimizer", "scheduler")  # Loop back
workflow.add_edge("summarizer", END)          # Summarizer is the final step

graph = workflow.compile(checkpointer=MemorySaver())
print("Graph compiled! Ready to run.")

Graph compiled! Ready to run.


## Project Presets
**Pick one of the presets below, or define your own!**

| # | Project | Team Size |
|---|---------|----------|
| A | E-commerce Mobile App | 3 people |
| B | AI Chatbot for Customer Support | 4 people |
| C | Internal Analytics Dashboard | 3 people |
| D | Security Audit & Refactor | 3 people |

In [6]:
# ══════════════════════════════════════════════════
# PRESET A: E-commerce Mobile App
# ══════════════════════════════════════════════════
preset_ecommerce = {
    "project_description": """
        Build a cross-platform mobile e-commerce app (iOS & Android) with:
        - User authentication (OAuth + JWT)
        - Product catalog with search & filters
        - Shopping cart and checkout
        - Stripe payment integration
        - Push notifications for order updates
        Timeline: 8 weeks.
    """,
    "team": Team(team_members=[
        TeamMember(name="Sofia", role="Mobile Lead", skills=["React Native", "iOS", "Android"], seniority="Senior"),
        TeamMember(name="James", role="Backend Dev", skills=["Python", "FastAPI", "PostgreSQL", "Stripe"], seniority="Mid"),
        TeamMember(name="Priya", role="QA Engineer", skills=["Testing", "Appium", "Jest"], seniority="Junior"),
    ])
}

# ══════════════════════════════════════════════════
# PRESET B: AI Chatbot for Customer Support
# ══════════════════════════════════════════════════
preset_chatbot = {
    "project_description": """
        Build an AI-powered customer support chatbot that:
        - Integrates with existing CRM (Salesforce)
        - Uses RAG (Retrieval-Augmented Generation) over product docs
        - Handles escalation to human agents
        - Supports 3 languages: English, Spanish, French
        - Has an admin dashboard for monitoring conversations
        Timeline: 6 weeks.
    """,
    "team": Team(team_members=[
        TeamMember(name="Alice", role="AI Engineer", skills=["Python", "LangChain", "OpenAI", "RAG"], seniority="Senior"),
        TeamMember(name="Bob", role="Backend Dev", skills=["Python", "FastAPI", "Salesforce API"], seniority="Mid"),
        TeamMember(name="Charlie", role="Frontend Dev", skills=["React", "TypeScript", "Dashboard UI"], seniority="Mid"),
        TeamMember(name="Diana", role="QA & Localization", skills=["Testing", "Spanish", "French"], seniority="Junior"),
    ])
}

# ══════════════════════════════════════════════════
# PRESET C: Internal Analytics Dashboard
# ══════════════════════════════════════════════════
preset_dashboard = {
    "project_description": """
        Build an internal analytics dashboard for the sales team:
        - Real-time KPI tracking (revenue, churn, MRR)
        - Connects to Snowflake data warehouse
        - Role-based access control (admin, manager, rep)
        - Automated weekly PDF reports via email
        - Mobile-responsive design
        Timeline: 5 weeks.
    """,
    "team": Team(team_members=[
        TeamMember(name="Leo", role="Data Engineer", skills=["Python", "Snowflake", "dbt", "SQL"], seniority="Senior"),
        TeamMember(name="Maya", role="Frontend Dev", skills=["React", "D3.js", "Tailwind", "Charts"], seniority="Mid"),
        TeamMember(name="Sam", role="DevOps", skills=["AWS", "Docker", "CI/CD", "Email automation"], seniority="Mid"),
    ])
}

# ══════════════════════════════════════════════════
# PRESET D: Security Audit & Refactor
# ══════════════════════════════════════════════════
preset_security = {
    "project_description": """
        Perform a security audit and refactor of an existing Node.js API:
        - Penetration testing and vulnerability assessment
        - Fix OWASP Top 10 vulnerabilities
        - Implement proper auth (OAuth2, MFA)
        - Add rate limiting, input validation, audit logs
        - Update all dependencies to latest secure versions
        Timeline: 4 weeks.
    """,
    "team": Team(team_members=[
        TeamMember(name="Nadia", role="Security Engineer", skills=["Penetration Testing", "OWASP", "OAuth2"], seniority="Senior"),
        TeamMember(name="Tom", role="Backend Dev", skills=["Node.js", "TypeScript", "API Security"], seniority="Mid"),
        TeamMember(name="Eva", role="DevOps", skills=["AWS", "Docker", "Monitoring", "Audit Logging"], seniority="Mid"),
    ])
}

print("4 project presets loaded!")
print("Available presets: preset_ecommerce, preset_chatbot, preset_dashboard, preset_security")

4 project presets loaded!
Available presets: preset_ecommerce, preset_chatbot, preset_dashboard, preset_security


## Run the Workflow
**Change `chosen_preset` to switch projects!**

In [7]:
# CHANGE THIS to switch projects:
# Options: preset_ecommerce | preset_chatbot | preset_dashboard | preset_security
chosen_preset = preset_chatbot

# Build initial state from chosen preset
init_state = {
    **chosen_preset,
    "iteration_number": 0,
    "max_iteration": 2,
    "insights": [],
    "project_risk_score_iterations": [],
    "final_summary": ""
}

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("Starting PM Workflow...")
print(f"Project: {init_state['project_description'].strip()[:80]}...\n")

final_state = graph.invoke(init_state, config=config)

print("\n Workflow complete!")

Starting PM Workflow...
Project: Build an AI-powered customer support chatbot that:
        - Integrates with exi...

Node: Scoper — decomposing project into tasks...


OutputParserException: Failed to parse TaskList from completion {"tasks": [{"task_name": "Integrate RAG with OpenAI", "task_description": "Configure and integrate RAG model with OpenAI API for retrieval augmentation", "estimated_day": 2, "required_skill": ["Python", "LangChain", "OpenAI"]}, {"task_name": "Setup Salesforce Integration", "task_description": "Connect chatbot to Salesforce CRM using API and retrieve customer data", "estimated_day": 3, "required_skill": ["Python", "FastAPI", "Salesforce API"]}, {"task_name": "Develop Chatbot UI", "task_description": "Build conversational interface using React, TypeScript, and dashboard UI components", "estimated_day": 4, "required_skill": ["React", "TypeScript", "Dashboard UI"]}, {"task_name": "Implement Language Support", "task_description": "Translate chatbot to support English, Spanish, and French languages", "estimated_day": 2, "required_skill": ["Testing", "Spanish", "French"]}, {"task_name": "Develop Admin Dashboard", "task_descriptio

## Results & Visualizations

In [ ]:
# ── AI-Generated Summary Report ──
display(Markdown("---"))
display(Markdown(final_state['final_summary']))
display(Markdown("---"))

---

**Project Summary Report**
==========================

### 1. **Executive Summary**
The AI-powered customer support chatbot project aims to integrate with Salesforce, utilize RAG over product docs, handle escalation to human agents, and support three languages. The project timeline is six weeks, and the team consists of four members with varying skill levels. The project's risk score has improved from 30 to 24, but there are still key risks that need to be addressed.

### 2. **Key Risks & Mitigations**
The top three risks for the project are:
1. **Implement Multilingual Support**: Risk score 8/10. Junior team member Diana is allocated to this task, which may lead to potential quality issues due to lack of experience. **Mitigation**: Provide additional guidance and support to Diana, and consider pairing her with a senior team member for review and feedback.
2. **Test and Debug**: Risk score 7/10. Junior team member Diana is allocated to this task, which may lead to potential quality issues due to lack of experience, and the task has a high estimated duration of 4 days. **Mitigation**: Reallocate this task to a senior team member, such as Alice, to ensure thorough and efficient testing and debugging.
3. **Design Admin Dashboard**: Risk score 4/10. Mid-level team member Charlie is allocated to this task, and the required skills match, but the task has a relatively high estimated duration of 3 days and may cause delays if not managed properly. **Mitigation**: Closely monitor Charlie's progress and provide additional resources or support if needed to ensure the task is completed on time.

### 3. **Team Workload Balance**
The team workload balance is a concern, as:
* Bob (Backend Dev) has a heavy workload with tasks such as Integrate Salesforce API, Develop Escalation Mechanism, Implement Admin Dashboard Backend, and Deploy and Monitor.
* Diana (QA & Localization) has a significant workload with tasks such as Implement Multilingual Support and Test and Debug, which may be challenging due to her junior level.
* Alice (AI Engineer) has a relatively light workload, with only two tasks: Setup Project Structure and Implement RAG Model.

### 4. **Recommended Next Steps**
To ensure the project's success, the following concrete actions are recommended:
1. **Reallocate the Test and Debug task**: Move this task from Diana to Alice to ensure thorough and efficient testing and debugging.
2. **Provide additional guidance and support to Diana**: Offer guidance and support to Diana for the Implement Multilingual Support task to mitigate the risk of potential quality issues.
3. **Closely monitor Charlie's progress**: Monitor Charlie's progress on the Design Admin Dashboard task and provide additional resources or support if needed to ensure the task is completed on time.

### 5. **Code & Architecture Improvements**
To improve the LangGraph agent code, the following suggestions are made:
1. **Implement automated testing**: Develop automated tests for the RAG model to ensure its accuracy and efficiency.
2. **Use a more efficient data storage solution**: Consider using a more efficient data storage solution, such as a graph database, to improve the performance of the chatbot.
3. **Add error handling and logging**: Implement robust error handling and logging mechanisms to ensure that errors are properly handled and logged, allowing for easier debugging and maintenance.

---

In [ ]:
# ── Risk Score Trend ──
risk_history = final_state['project_risk_score_iterations']
display(Markdown(f"### Risk Score History"))
for i, score in enumerate(risk_history):
    bar = "█" * (score // 3)
    label = "🔴 High" if score > 30 else "🟡 Medium" if score > 15 else "🟢 Low"
    print(f"  Iteration {i+1}: {bar} {score} — {label}")

# ── Team Structure ──
display(Markdown(f"\n### Team Structure"))
for m in final_state['team'].team_members:
    tasks_assigned = sum(
        1 for a in final_state['task_allocations'].task_allocations
        if a.team_member.name == m.name
    )
    print(f"  └── {m.name} ({m.role}, {m.seniority}) — {tasks_assigned} task(s) assigned")

# ── Gantt Chart ──
display(Markdown(f"\n### Project Schedule (Gantt)"))
sched_data = []
start_date_base = datetime.now()
alloc_map = {a.task.task_name: a.team_member.name for a in final_state['task_allocations'].task_allocations}

for item in final_state['schedule'].schedule:
    t_name = item.task.task_name
    assignee = alloc_map.get(t_name, "Unassigned")
    s_date = start_date_base + timedelta(days=item.start_day)
    e_date = start_date_base + timedelta(days=item.end_day)
    sched_data.append({
        "Task": t_name,
        "Start": s_date.strftime("%Y-%m-%d"),
        "Finish": e_date.strftime("%Y-%m-%d"),
        "Assignee": assignee,
        "Duration": item.end_day - item.start_day
    })

df = pd.DataFrame(sched_data).sort_values("Start")

if not df.empty:
    fig = px.timeline(
        df, 
        x_start="Start", 
        x_end="Finish", 
        y="Task", 
        color="Assignee",
        title=f"Project Schedule — {final_state['project_description'].strip()[:50]}...",
        hover_data=["Duration"]
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(height=max(400, len(sched_data) * 40))
    fig.show()
else:
    print("No schedule data to plot.")

### Risk Score History

  Iteration 1: ██████████ 30 — 🟡 Medium
  Iteration 2: ████████ 24 — 🟡 Medium



### Team Structure

  └── Alice (AI Engineer, Senior) — 2 task(s) assigned
  └── Bob (Backend Dev, Mid) — 4 task(s) assigned
  └── Charlie (Frontend Dev, Mid) — 1 task(s) assigned
  └── Diana (QA & Localization, Junior) — 2 task(s) assigned



### Project Schedule (Gantt)

In [ ]:
# ── Risk Breakdown Table ──
from IPython.display import HTML

risks_sorted = sorted(final_state['risks'].risks, key=lambda x: x.score, reverse=True)

def score_color(score):
    if score >= 7: return "#ff4d4d", "🔴"
    if score >= 4: return "#ffa500", "🟡"
    return "#4caf50", "🟢"

tbody = ""
for i, r in enumerate(risks_sorted):
    color, emoji = score_color(r.score)
    bg = "#f9f9f9" if i % 2 == 0 else "#ffffff"
    tbody += (
        f'<tr style="background:{bg};">'
        f'<td style="padding:10px 14px; font-weight:500; color:#111111;">{r.task_name}</td>'
        f'<td style="padding:10px 14px; text-align:center;">'
        f'<span style="background:{color}; color:white; padding:3px 10px; border-radius:12px; font-weight:bold;">'
        f'{emoji} {r.score}/10</span></td>'
        f'<td style="padding:10px 14px; color:#333333;">{r.reason}</td>'
        f'</tr>'
    )

html = (
    "<h3 style='color:#ffffff;'>⚠️ Risk Breakdown</h3>"
    '<table style="border-collapse:collapse; width:100%; font-family:sans-serif; font-size:14px; background:#ffffff;">'
    "<thead>"
    '<tr style="background:#2d2d2d; color:#ffffff;">'
    '<th style="padding:12px 14px; text-align:left; color:#ffffff;">Task</th>'
    '<th style="padding:12px 14px; text-align:center; width:120px; color:#ffffff;">Risk Score</th>'
    '<th style="padding:12px 14px; text-align:left; color:#ffffff;">Reason</th>'
    "</tr></thead>"
    f"<tbody>{tbody}</tbody>"
    "</table>"
)

display(HTML(html))

Task,Risk Score,Reason
Implement Multilingual Support,🔴 8/10,"Junior team member Diana is allocated to this task, which may lead to potential quality issues due to lack of experience."
Test and Debug,🔴 7/10,"Junior team member Diana is allocated to this task, which may lead to potential quality issues due to lack of experience, and the task has a high estimated duration of 4 days."
Design Admin Dashboard,🟡 4/10,"Mid-level team member Charlie is allocated to this task, and the required skills match, but the task has a relatively high estimated duration of 3 days and may cause delays if not managed properly."
Implement Admin Dashboard Backend,🟢 3/10,"Mid-level team member Bob is allocated to this task, and the required skills match, but the task is scheduled after the Design Admin Dashboard task, which may cause delays if the design task is not completed on time."
Integrate Salesforce API,🟢 2/10,"Mid-level team member Bob is allocated to this task, and the required skills match, but the task has a relatively short estimated duration of 2 days and may be completed quickly."
